In [2]:
import polars as pl
import datetime as dt
import os
from pathlib import Path

In [3]:
L2DATA_PATH = "/data/xujiayi/xjy/l2/test"

In [4]:
def normalize_date(date: dt.date | dt.datetime | str) -> str:
    if isinstance(date, (dt.datetime, dt.date)):
        return date.strftime("%Y%m%d")
    return str(date).replace("-", "").replace(".", "").strip("/")

In [5]:
date = normalize_date('20250930')

filepath = Path(L2DATA_PATH)/"raw"/date

outpath = Path(L2DATA_PATH)/"proc"/date
outpath.mkdir(parents=True, exist_ok=True)

#### 深交所三表整理

In [9]:
szcj_merged = pl.read_parquet(filepath/'szcj.pq').filter(pl.col('MDStreamID')==11).drop(['MDStreamID','SecurityIDSource','LocalTime','SeqNo']).rename({
    'LastPx':'Price',
    'LastQty':'OrderQty'
}).with_columns([
    pl.col('TransactTime').str.to_time(format="%H:%M:%S%.3f"),
    pl.when(pl.col('BidApplSeqNum')>pl.col('OfferApplSeqNum')).then(pl.lit(1)).otherwise(pl.lit(-1)).alias('Side')
])
szcj = szcj_merged.filter(pl.col('ExecType')==70).drop('ExecType')
szcancel = szcj_merged.filter(pl.col('ExecType')==52).drop('ExecType')

In [10]:
szcj.write_parquet(outpath/'szcj.pq',compression="gzip")
szcancel.write_parquet(outpath/'szcancel.pq',compression="gzip")

In [11]:
szwt = pl.read_parquet(filepath/'szwt.pq').filter(pl.col('MDStreamID')==11).drop(['MDStreamID','SecurityIDSource','LocalTime','SeqNo','OrdType'])
szwt = szwt.with_columns([
    pl.col('Side').replace(49,1).replace(50,-1).cast(pl.Int8),
    pl.col('TransactTime').str.to_time(format="%H:%M:%S%.3f")
])

In [12]:
szwt.write_parquet(outpath/'szwt.pq',compression="gzip")

#### 上交所三表整理

In [6]:
sh = pl.read_parquet(filepath/'sh.pq').drop(['TradeMoney','LocalTime','SeqNo']).rename({
    'BizIndex':'ApplSeqNum',
    'Channel':'ChannelNo',
    'TickTime':'TransactTime',
    'Qty':'OrderQty'
}).with_columns([
    pl.col('TransactTime').str.to_time(format="%H:%M:%S%.3f"),
])
sh

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Type,BuyOrderNO,SellOrderNO,Price,OrderQty,TickBSFlag
i64,i64,i64,time,str,i64,i64,f64,i64,str
1,1,751028,06:00:00.110,"""S""",0,0,0.0,0,"""SUSP"""
2,1,751900,06:00:00.110,"""S""",0,0,0.0,0,"""SUSP"""
3,1,751004,06:00:00.110,"""S""",0,0,0.0,0,"""SUSP"""
4,1,751019,06:00:00.110,"""S""",0,0,0.0,0,"""SUSP"""
5,1,751992,06:00:00.110,"""S""",0,0,0.0,0,"""SUSP"""
…,…,…,…,…,…,…,…,…,…
28262536,6,560820,15:05:00.560,"""S""",0,0,0.0,0,"""ENDTR"""
28262537,6,588130,15:05:00.560,"""S""",0,0,0.0,0,"""ENDTR"""
28262538,6,603210,15:05:00.560,"""S""",0,0,0.0,0,"""ENDTR"""


In [7]:
shwt_added = sh.filter(pl.col('Type')=='A').drop('Type').with_columns([
    pl.when(pl.col('BuyOrderNO')!=0).then(pl.col('BuyOrderNO')).otherwise(pl.col('SellOrderNO')).alias('OrderNo'),
    pl.when(pl.col('BuyOrderNO')!=0).then(pl.lit(1)).otherwise(pl.lit(-1)).alias('Side'),
]).drop(['BuyOrderNO','SellOrderNO'])
shwt_added 

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Price,OrderQty,TickBSFlag,OrderNo,Side
i64,i64,i64,time,f64,i64,str,i64,i32
660,2,688545,09:15:01.970,36.01,300,"""B""",12330,1
661,2,688545,09:15:02.550,41.42,1704,"""S""",20176,-1
662,2,688545,09:15:02.550,45.23,9,"""S""",21026,-1
663,2,688545,09:15:02.550,37.02,1000,"""B""",21338,1
664,2,688545,09:15:02.680,36.71,1000,"""B""",21373,1
…,…,…,…,…,…,…,…,…
31571741,5,603202,14:59:42.510,98.73,100,"""S""",19318579,-1
31571742,5,603202,14:59:45,96.87,100,"""B""",19320909,1
31571743,5,603202,14:59:52.550,96.77,200,"""S""",19329054,-1


In [10]:
shwt_added.filter(pl.col('SecurityID')==688545).filter(pl.col('OrderNo')==23083)

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Price,OrderQty,TickBSFlag,OrderNo,Side
i64,i64,i64,time,f64,i64,str,i64,i32
666,2,688545,09:15:02.680,39.9,1889,"""S""",23083,-1


In [13]:
sh.filter(
    (pl.col('Type')=='T')
).filter(pl.col('SecurityID')==688545).filter(pl.col('SellOrderNO')==23083)

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Type,BuyOrderNO,SellOrderNO,Price,OrderQty,TickBSFlag
i64,i64,i64,time,str,i64,i64,f64,i64,str
6119845,2,688545,09:48:38.420,"""T""",4004139,23083,39.9,285,"""B"""
6119853,2,688545,09:48:38.450,"""T""",4004145,23083,39.9,200,"""B"""
6119914,2,688545,09:48:38.470,"""T""",4004188,23083,39.9,500,"""B"""
6119915,2,688545,09:48:38.470,"""T""",4004189,23083,39.9,904,"""B"""


In [11]:
shcj = sh.filter(
    (pl.col('Type')=='T') | (pl.col('Type')=='D')
).join(
    shwt_added.select(['ChannelNo','OrderNo','ApplSeqNum','SecurityID']),  # 获取买单的频道内索引
    left_on=['ChannelNo','BuyOrderNO','SecurityID'],
    right_on=['ChannelNo','OrderNo','SecurityID'],
    how='left',
    suffix='_buy'
).join(
    shwt_added.select(['ChannelNo','OrderNo','ApplSeqNum','SecurityID']),  # 获取卖单的频道内索引
    left_on=['ChannelNo','SellOrderNO','SecurityID'],
    right_on=['ChannelNo','OrderNo','SecurityID'],
    how='left',
    suffix='_sell'
).rename({
    'ApplSeqNum_buy':'BidApplSeqNum',
    'ApplSeqNum_sell':'OfferApplSeqNum'
}).with_columns([
    pl.when(pl.col('Type')=='D').then(pl.col('BidApplSeqNum').fill_null(0)).otherwise(pl.col('BidApplSeqNum').fill_null(pl.col('ApplSeqNum'))),
    pl.when(pl.col('Type')=='D').then(pl.col('OfferApplSeqNum').fill_null(0)).otherwise(pl.col('OfferApplSeqNum').fill_null(pl.col('ApplSeqNum'))),
]).with_columns([                                               # 判断买卖方向
    pl.when(pl.col('BidApplSeqNum')>pl.col('OfferApplSeqNum')).then(pl.lit(1)).otherwise(pl.lit(-1)).alias('Side')
]).drop(['BuyOrderNO','SellOrderNO'])
shcj

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Type,Price,OrderQty,TickBSFlag,BidApplSeqNum,OfferApplSeqNum,Side
i64,i64,i64,time,str,f64,i64,str,i64,i64,i32
810,2,688545,09:15:11.450,"""D""",38.17,12717,"""S""",0,699,-1
860,2,688545,09:15:17.280,"""D""",37.49,267,"""S""",0,859,-1
870,2,688545,09:15:18.380,"""D""",45.23,350,"""S""",0,868,-1
899,2,688545,09:15:28.010,"""D""",37.7,1325,"""B""",894,0,1
901,2,688545,09:15:29.850,"""D""",38.0,778,"""S""",0,900,-1
…,…,…,…,…,…,…,…,…,…,…
31571810,5,603202,15:00:01.690,"""T""",98.75,400,"""N""",31571725,31571710,1
31571811,5,603202,15:00:01.690,"""T""",98.75,100,"""N""",31571733,31571710,1
31571812,5,603202,15:00:01.690,"""T""",98.75,1000,"""N""",31571735,31571710,1


In [159]:
shcancel = shcj.filter(pl.col('Type')=='D').drop('Type')
shcancel

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Price,OrderQty,TickBSFlag,BidApplSeqNum,OfferApplSeqNum,Side
i64,i64,i64,time,f64,i64,str,i64,i64,i32
810,2,688545,09:15:11.450,38.17,12717,"""S""",0,699,-1
860,2,688545,09:15:17.280,37.49,267,"""S""",0,859,-1
870,2,688545,09:15:18.380,45.23,350,"""S""",0,868,-1
899,2,688545,09:15:28.010,37.7,1325,"""B""",894,0,1
901,2,688545,09:15:29.850,38.0,778,"""S""",0,900,-1
…,…,…,…,…,…,…,…,…,…
28086978,6,516510,14:59:59.930,1.673,280300,"""B""",28085229,0,1
28086979,6,516510,14:59:59.930,1.67,280300,"""B""",28048955,0,1
28086981,6,513700,14:59:59.950,0.77,3000,"""B""",26342070,0,1


In [160]:
shcancel.write_parquet(outpath/'shcancel.pq',compression="gzip")

In [161]:
shcj = shcj.filter(pl.col('Type')=='T').drop('Type')
shcj

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Price,OrderQty,TickBSFlag,BidApplSeqNum,OfferApplSeqNum,Side
i64,i64,i64,time,f64,i64,str,i64,i64,i32
1210,2,688545,09:25:00.010,38.75,50,"""N""",1021,869,1
1211,2,688545,09:25:00.010,38.75,393,"""N""",1021,953,1
1212,2,688545,09:25:00.010,38.75,200,"""N""",1021,983,1
1213,2,688545,09:25:00.010,38.75,157,"""N""",1021,985,1
1214,2,688545,09:25:00.010,38.75,1000,"""N""",1052,985,1
…,…,…,…,…,…,…,…,…,…
31571810,5,603202,15:00:01.690,98.75,400,"""N""",31571725,31571710,1
31571811,5,603202,15:00:01.690,98.75,100,"""N""",31571733,31571710,1
31571812,5,603202,15:00:01.690,98.75,1000,"""N""",31571735,31571710,1


In [162]:
shcj.write_parquet(outpath/'shcj.pq',compression="gzip")

In [ ]:
shwt_added = shwt_added.drop('OrderNo')
shwt_added

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Price,OrderQty,TickBSFlag,Side
i64,i64,i64,time,f64,i64,str,i32
660,2,688545,09:15:01.970,36.01,300,"""B""",1
661,2,688545,09:15:02.550,41.42,1704,"""S""",-1
662,2,688545,09:15:02.550,45.23,9,"""S""",-1
663,2,688545,09:15:02.550,37.02,1000,"""B""",1
664,2,688545,09:15:02.680,36.71,1000,"""B""",1
…,…,…,…,…,…,…,…
31571741,5,603202,14:59:42.510,98.73,100,"""S""",-1
31571742,5,603202,14:59:45,96.87,100,"""B""",1
31571743,5,603202,14:59:52.550,96.77,200,"""S""",-1


: 

In [31]:
def RestoreOrder(wt, cj):
    
    cj = cj.sort(["ChannelNo", "ApplSeqNum", "SecurityID", "TransactTime"])

    # 剔除集合竞价
    cj_df = cj.filter(~((pl.col("TransactTime") <= pl.time(9, 25)) | (pl.col("TransactTime") >= pl.time(14, 57))))

    # 1. 从成交表提取买卖双方订单并汇总
    cj_buy = cj_df.select([
        "ChannelNo", "BidApplSeqNum", "SecurityID", "OrderQty", "Price", "TransactTime",
    ]).rename({"BidApplSeqNum": "ApplSeqNum"}).with_columns(pl.lit(1).alias("Side"))

    cj_sell = cj_df.select([
        "ChannelNo", "OfferApplSeqNum", "SecurityID", "OrderQty", "Price", "TransactTime",
    ]).rename({"OfferApplSeqNum": "ApplSeqNum"}).with_columns(pl.lit(-1).alias("Side"))

    cj_all = pl.concat([cj_buy, cj_sell])
    cj_summary = cj_all.group_by(["ChannelNo", "ApplSeqNum", "SecurityID", "Side"]).agg([
        pl.sum("OrderQty"),
        pl.last("Price"),  # 一笔主动单可能同时吃掉多笔挂单
        pl.last("TransactTime")
    ])

    # 2. 使用反连接和半连接分离三种情况
    # 部分成交：同时存在于委托和成交（inner join）
    partial = wt.join(
        cj_summary.select(["ChannelNo", "ApplSeqNum", "SecurityID", "Side", "OrderQty"]),
        on=["ChannelNo", "ApplSeqNum", "SecurityID", "Side"],
        how="inner"
    ).with_columns([
        (pl.col("OrderQty") + pl.col("OrderQty_right")).alias("OrderQty"),
        pl.lit("主动部分成交").alias("OrderStatus")
    ]).drop("OrderQty_right")

    # 未成交：存在于委托但不存在于成交（anti join）
    untouched = wt.join(
        cj_summary.select(["ChannelNo", "ApplSeqNum", "SecurityID", "Side"]),
        on=["ChannelNo", "ApplSeqNum", "SecurityID", "Side"],
        how="anti"
    ).with_columns(
        pl.lit("挂单被动成交").alias("OrderStatus")
    )
    untouched = untouched.select(partial.columns)

    # 完全成交：存在于成交但不存在于委托（anti join）
    new = cj_summary.join(
        wt.select(["ChannelNo", "ApplSeqNum", "SecurityID", "Side"]),
        on=["ChannelNo", "ApplSeqNum", "SecurityID", "Side"],
        how="anti"
    ).with_columns([
        pl.lit("主动完全成交").alias("OrderStatus"),
    ])
    new = new.select(partial.columns)

    # 3. 合并所有订单
    init_order = pl.concat([partial, untouched, new])
    return init_order

In [32]:
shwt = RestoreOrder(shwt_added, shcj)
shwt

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Price,OrderQty,Side,OrderStatus
i64,i64,i64,time,f64,i64,i32,str
13701336,6,603270,10:47:51.190,26.29,200,-1,"""主动部分成交"""
5911366,1,603888,09:46:49.180,19.69,1800,-1,"""主动部分成交"""
7479132,2,603328,09:55:22.950,11.94,1400,1,"""主动部分成交"""
3800915,3,600211,09:38:40.310,46.35,5800,1,"""主动部分成交"""
16151645,3,603626,10:58:14.270,14.46,16200,1,"""主动部分成交"""
…,…,…,…,…,…,…,…
15259738,3,600522,10:49:01.590,18.73,400,-1,"""主动完全成交"""
5528821,6,601860,09:47:46.300,2.82,400,1,"""主动完全成交"""
5991244,5,600895,09:46:40.250,51.96,600,-1,"""主动完全成交"""


In [33]:
shwt.drop('OrderStatus').write_parquet(outpath/'shwt.pq',compression="gzip")

#### 检验还原委托的正确性

In [109]:
def get_shot(exchange):
    shot = pl.read_parquet(Path(L2DATA_PATH)/"raw"/f"{date}"/f"{exchange}shot.pq")
    cols = ['SecurityID','UpdateTime'] + ','.join([f'BidPrice{i},BidVolume{i},AskPrice{i},AskVolume{i}' for i in range(1, 11)]).split(',')
    shot = shot.select(cols)
    closing_shot = shot.sort(['SecurityID', 'UpdateTime']).group_by('SecurityID').last()
    closing_orderbook = pl.concat([
        closing_shot.select(
            pl.col("SecurityID"),
            pl.lit(i).alias("Level"),

            pl.col(f"BidPrice{i}")
            .cast(pl.Float64, strict=False)
            .alias("BidPrice"),

            pl.col(f"BidVolume{i}")
            .cast(pl.Float64, strict=False).cast(pl.Int64, strict=False)
            .alias("BidQty"),

            pl.col(f"AskPrice{i}")
            .cast(pl.Float64, strict=False)
            .alias("AskPrice"),

            pl.col(f"AskVolume{i}")
            .cast(pl.Float64, strict=False).cast(pl.Int64, strict=False)
            .alias("AskQty"),
        )
        for i in range(1, 11)
    ]).sort(["SecurityID", "Level"])
    return closing_orderbook

In [55]:
def get_close_orderbook(
    shwt: pl.DataFrame,
    shcj: pl.DataFrame,
    shcd: pl.DataFrame,
    topn: int = 10,
):
    # 1. 委托订单表
    orders = (
        shwt
        .select([
            "ChannelNo",
            "SecurityID",
            "ApplSeqNum",
            "Price",
            "OrderQty",
            "Side",
        ])
    )

    cj = pl.concat([shcj, shcd])

    # 2. 买方订单成交扣减
    buy_filled = (
        cj
        .select([
            "ChannelNo",
            "SecurityID",
            pl.col("BidApplSeqNum").alias("ApplSeqNum"),
            pl.col("OrderQty").alias("DealQty"),
        ])
        .with_columns(pl.lit(1).alias('Side'))
    )

    # 3. 卖方订单成交扣减
    sell_filled = (
        cj
        .select([
            "ChannelNo",
            "SecurityID",
            pl.col("OfferApplSeqNum").alias("ApplSeqNum"),
            pl.col("OrderQty").alias("DealQty"),
        ])
        .with_columns(pl.lit(-1).alias('Side'))
    )

    # 4. 每张订单的累计成交数量
    filled = (
        pl.concat([buy_filled, sell_filled])
        .group_by(["ChannelNo", "SecurityID", "Side", "ApplSeqNum"])
        .agg(
            pl.col("DealQty").sum().alias("FilledQty")
        )
    )

    # 8. 计算每张订单剩余数量
    alive_orders = (
        orders
        .join(
            filled,
            on=["ChannelNo", "SecurityID", "Side", "ApplSeqNum"],
            how="left",
        )
        .with_columns([
            (
                pl.col("OrderQty")
                - pl.col("FilledQty").fill_null(0)
            ).alias("RemainQty")
        ])
        .filter(pl.col("RemainQty") > 0)
    )

    # 9. 聚合成价格档位
    price_level = (
        alive_orders
        .group_by(["SecurityID", "Side", "Price"])
        .agg(
            pl.col("RemainQty").sum().alias("Qty")
        )
    )

    # 10. 买盘：价格从高到低
    bid_book = (
        price_level
        .filter(pl.col("Side") == 1)
        .with_columns(
            pl.col("Price")
            .rank(method="dense", descending=True)
            .over("SecurityID")
            .cast(pl.Int32)
            .alias("Level")
        )
        .filter(pl.col("Level") <= topn)
        .sort(["SecurityID", "Level"])
        .rename({
            "Price": "BidPrice",
            "Qty": "BidQty",
        })
        .select(["SecurityID", "Level", "BidPrice", "BidQty"])
    )

    # 11. 卖盘：价格从低到高
    ask_book = (
        price_level
        .filter(pl.col("Side") == -1)
        .with_columns(
            pl.col("Price")
            .rank(method="dense", descending=False)
            .over("SecurityID")
            .cast(pl.Int32)
            .alias("Level")
        )
        .filter(pl.col("Level") <= topn)
        .sort(["SecurityID", "Level"])
        .rename({
            "Price": "AskPrice",
            "Qty": "AskQty",
        })
        .select(["SecurityID", "Level", "AskPrice", "AskQty"])
    )

    # 12. 合并买卖盘
    close_book = (
        bid_book
        .join(
            ask_book,
            on=["SecurityID", "Level"],
            how="full",
            coalesce=True,
        )
        .sort(["SecurityID", "Level"])
    )

    return close_book, alive_orders

##### 检查深交所

In [110]:
sz_closing_orderbook = get_shot('sz')

In [ ]:
sz_closing_orderbook.filter(pl.col('SecurityID')==300575)

In [112]:
szcj = pl.read_parquet(outpath/'szcj.pq')
szwt = pl.read_parquet(outpath/'szwt.pq')
szcancel = pl.read_parquet(outpath/'szcancel.pq')

In [113]:
sz_close_book, sz_alive_orders = get_close_orderbook(szwt, szcj, szcancel, topn=10)

In [ ]:
sz_close_book.filter(pl.col('SecurityID')==300575)

##### 检查上交所

In [115]:
sh_closing_orderbook = get_shot('sh')

In [116]:
sh_closing_orderbook.filter(pl.col('SecurityID')==600000)

SecurityID,Level,BidPrice,BidQty,AskPrice,AskQty
i64,i32,f64,i64,f64,i64
600000,1,11.9,254017,11.91,155700
600000,2,11.89,12700,11.92,181200
600000,3,11.88,11400,11.93,39300
600000,4,11.87,21500,11.94,69200
600000,5,11.86,22200,11.95,120871
600000,6,11.85,719800,11.96,166005
600000,7,11.84,133100,11.97,1256300
600000,8,11.83,233700,11.98,202000
600000,9,11.82,46200,11.99,20235400


In [117]:
shwt = pl.read_parquet(outpath/'shwt.pq')
shcj = pl.read_parquet(outpath/'shcj.pq')
shcancel = pl.read_parquet(outpath/'shcancel.pq')

In [122]:
sh_close_book, sh_alive_orders = get_close_orderbook(shwt, shcj, shcancel, topn=40)

In [123]:
sh_close_book.filter(pl.col('SecurityID')==600000)

SecurityID,Level,BidPrice,BidQty,AskPrice,AskQty
i64,i32,f64,i64,f64,i64
600000,1,13.27,27700,10.85,12600
600000,2,13.26,2000,11.22,6000
600000,3,13.22,2000,11.46,31400
600000,4,12.8,1000,11.49,2700
600000,5,12.66,13100,11.5,300
…,…,…,…,…,…
600000,36,11.95,1355840,12.09,167100
600000,37,11.94,9082189,12.1,375357
600000,38,11.93,7095039,12.11,206020
